In [1]:
%pip install -q -U gradio langchain langchain-cohere langchain-community langchain-text-splitters faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.7/402.7 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.3/573.3 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 

In [2]:

import gradio as gr
from getpass import getpass

from langchain_cohere import ChatCohere, CohereEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate

# STEP 1: Enter your Cohere API key
api_key = getpass("Enter your Cohere API key: ")

# STEP 2: Initialize models
embeddings = CohereEmbeddings(
    model="embed-english-v3.0",
    cohere_api_key=api_key
)

llm = ChatCohere(
    model="command-a-03-2025",
    cohere_api_key=api_key,
    temperature=0
)

# STEP 3: Create a prompt template
prompt = ChatPromptTemplate.from_template("""
You are a document question-answering assistant.

Answer the question using only the context below.
If the answer is not present in the document, clearly say
that the information was not found in the uploaded PDF.

Context:
{context}

Question:
{question}

Answer:
""")

# STEP 4: Store the vector database
vector_store = None


# STEP 5: Process the uploaded PDF
def process_pdf(pdf_file):

    global vector_store

    if pdf_file is None:
        return "Please upload a PDF file first."

    try:
        # Extract text from PDF
        loader = PyPDFLoader(pdf_file)
        documents = loader.load()

        # Remove empty pages
        documents = [
            doc for doc in documents
            if doc.page_content.strip()
        ]

        if not documents:
            vector_store = None
            return "No readable text was found in this PDF."

        # Split text into chunks
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=800,
            chunk_overlap=100
        )

        chunks = splitter.split_documents(documents)

        if not chunks:
            vector_store = None
            return "Could not create text chunks from this PDF."

        # Create embeddings and vector store
        vector_store = FAISS.from_documents(
            chunks,
            embeddings
        )

        return (
            "PDF processed successfully!\n"
            f"Pages loaded: {len(documents)}\n"
            f"Text chunks created: {len(chunks)}\n"
            "You can now ask questions about the PDF."
        )

    except Exception as e:
        vector_store = None
        return f"Error processing PDF: {str(e)}"


# STEP 6: Answer questions using RAG
def ask_question(question):

    if vector_store is None:
        return "Please upload a PDF and click Build Knowledge Base first."

    if not question or not question.strip():
        return "Please enter a question."

    try:
        # Retrieve relevant chunks
        retrieved_docs = vector_store.similarity_search(
            question,
            k=3
        )

        if not retrieved_docs:
            return "No relevant information was retrieved."

        context = "\n\n".join(
            doc.page_content for doc in retrieved_docs
        )

        # Generate answer
        messages = prompt.format_messages(
            context=context,
            question=question
        )

        response = llm.invoke(messages)

        # Show source pages
        source_pages = sorted(set(
            doc.metadata.get("page", 0) + 1
            for doc in retrieved_docs
        ))

        answer = response.content

        return (
            f"{answer}\n\n"
            f"Source PDF pages: {source_pages}"
        )

    except Exception as e:
        return f"Error answering question: {str(e)}"


# STEP 7: Build the Gradio interface
with gr.Blocks(title="PDF RAG Assistant") as demo:

    gr.Markdown("# PDF Question Answering with RAG")
    gr.Markdown(
        "Upload a PDF, build the knowledge base, "
        "and ask questions about its contents."
    )

    pdf_input = gr.File(
        label="Upload your PDF",
        file_types=[".pdf"],
        type="filepath"
    )

    build_button = gr.Button(
        "Build Knowledge Base",
        variant="primary"
    )

    status_output = gr.Textbox(
        label="Processing Status",
        interactive=False
    )

    question_input = gr.Textbox(
        label="Ask a question",
        placeholder="What are the main topics in this PDF?"
    )

    ask_button = gr.Button("Get Answer")

    answer_output = gr.Textbox(
        label="Answer and Source Pages",
        lines=10,
        interactive=False
    )

    build_button.click(
        fn=process_pdf,
        inputs=pdf_input,
        outputs=status_output
    )

    ask_button.click(
        fn=ask_question,
        inputs=question_input,
        outputs=answer_output
    )

    question_input.submit(
        fn=ask_question,
        inputs=question_input,
        outputs=answer_output
    )


# STEP 8: Launch the application
demo.launch(share=True)


/tmp/ipykernel_1300/3872427946.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Enter your Cohere API key: ··········
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7f3461378eda6a60c3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
